# About This notebook

This is notebook imports BLS Wage datasets and prepares them for insertion into a postgresql database for further analysis. 

When completed, the idea is that this notebook is run once, transferring the data in as raw and pristine of a fashion as possible to the postgresql database.

If the data in the database is found to be unsatisfactory, the associated database tables would need to be dropped, this file would need to be adapted as necessary, and then run again. 

## Link to Data

[https://www.bls.gov/oes/](https://www.bls.gov/oes/)

# Project Setup

## Import and Verifications

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sqlalchemy

In [2]:
print(sqlalchemy.__version__)

2.0.48


In [3]:
from sqlalchemy import create_engine
from sqlalchemy import text

## Create SQL Engine and Test Connection

In [4]:
engine = create_engine('postgresql+psycopg2://postgres@localhost/train_reward_compare')

In [5]:
with engine.connect() as conn:
    result = conn.execute(text("select count(*) from onet.occupation_data"))
    print(result.fetchone())

(1016,)


## Import BLS Files

In [6]:
file_nat = '../../raw-data/bls-wage/oesm24nat/national_M2024_dl.xlsx'
file_sta = '../../raw-data/bls-wage/oesm24st/state_M2024_dl.xlsx'
file_met = '../../raw-data/bls-wage/oesm24ma/MSA_M2024_dl.xlsx'
file_bos = '../../raw-data/bls-wage/oesm24ma/BOS_M2024_dl.xlsx'
file_nbs = '../../raw-data/bls-wage/oesm24in4/natsector_M2024_dl.xlsx'

In [7]:
nat = pd.ExcelFile(file_nat)
sta = pd.ExcelFile(file_sta)
met = pd.ExcelFile(file_met)
bos = pd.ExcelFile(file_bos)
nbs = pd.ExcelFile(file_nbs)

## Test Data in Each Imported File

In [8]:
print(nat.sheet_names)
print(sta.sheet_names)
print(met.sheet_names)
print(bos.sheet_names)
print(nbs.sheet_names)

['national_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['state_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['MSA_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['BOS_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['Natsector_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']


# Rough Wrangling

## National Data

### Rough Wrangle National BLS Data

In [9]:
nat.parse('national_M2024_dl').head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN


In [10]:
nat_data = nat.parse('national_M2024_dl')

In [11]:
nat_data.columns = nat_data.columns.str.lower()

In [12]:
nat_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN


### Rough Wrangle National Field Descriptions

In [13]:
nat_desc_pre = nat.parse('Field Descriptions')

In [14]:
nat_desc_pre.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [15]:
nat_desc_pre = nat_desc_pre.drop('Unnamed: 2', axis=1)

In [16]:
drop_index = nat_desc_pre.index[0:8]

In [17]:
nat_desc_pre_dropped = nat_desc_pre.drop(drop_index)

In [18]:
nat_desc_pre_dropped.head()

,May 2024 OEWS Estimates,Unnamed: 1
8,Field,Field Description
9,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
10,area_title,Area name
11,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
12,prim_state,"The primary state for the given area. ""US"" is ..."


In [19]:
nat_desc_pre_new_header = nat_desc_pre_dropped.iloc[0]

In [20]:
nat_desc_pre_new = nat_desc_pre_dropped[1:]

In [21]:
nat_desc_pre_new.columns = nat_desc_pre_new_header

In [22]:
nat_desc_pre_new = nat_desc_pre_new.reset_index(drop=True)

In [23]:
nat_desc_pre_new.index.name = None

In [24]:
nat_desc_pre_new = nat_desc_pre_new.rename_axis(None, axis=1)

In [25]:
nat_desc_pre_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


In [26]:
nat_desc_pre_new.tail(10)

,Field,Field Description
28,a_pct75,Annual 75th percentile wage
29,a_pct90,Annual 90th percentile wage
30,annual,"Contains ""TRUE"" if only annual wages are relea..."
31,hourly,"Contains ""TRUE"" if only hourly wages are relea..."
32,NaN,NaN
33,Notes:,NaN
34,* = indicates that a wage estimate is not ava...,NaN
35,** = indicates that an employment estimate is...,NaN
36,# = indicates a wage equal to or greater than...,NaN
37,~ =indicates that the percent of establishment...,NaN


#### Split Descriptions and Notes

In [27]:
notes_df = nat_desc_pre_new.iloc[32:,:]

In [28]:
nat_desc = nat_desc_pre_new.iloc[:32, :]

In [29]:
nat_desc

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...
5,naics_title,North American Industry Classification System ...
6,i_group,Industry level. Indicates cross-industry or NA...
7,own_code,Ownership type: 1= Federal Government; 2= Stat...
8,occ_code,The 6-digit Standard Occupational Classificati...
9,occ_title,SOC title or OEWS-specific title for the occup...


## Storing Notes in DataFrame

In [30]:
notes_df = notes_df.drop(columns=['Field Description'], axis=1)

In [31]:
notes_df = notes_df.iloc[2:,:]

In [32]:
notes_df = notes_df.rename(columns={'Field':'Notes'}) 

In [33]:
notes_df

,Notes
34,* = indicates that a wage estimate is not ava...
35,** = indicates that an employment estimate is...
36,# = indicates a wage equal to or greater than...
37,~ =indicates that the percent of establishment...


## State Data

### Rough Wrangle State Data

In [34]:
sta.parse('state_M2024_dl').head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.07,30.82,47.51,23520,30660,43830,64110,98810,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,48.39,68.5,98.03,51100,72870,100640,142480,203900,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.04,106.69,#,104950,130950,164400,221910,#,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,51.12,78.26,#,50410,74720,106330,162780,#,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18270,20950,26990,41760,63900,True,NaN


In [35]:
sta_data = sta.parse('state_M2024_dl')

In [36]:
sta_data.columns = sta_data.columns.str.lower()

In [37]:
sta_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.07,30.82,47.51,23520,30660,43830,64110,98810,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,48.39,68.5,98.03,51100,72870,100640,142480,203900,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.04,106.69,#,104950,130950,164400,221910,#,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,51.12,78.26,#,50410,74720,106330,162780,#,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18270,20950,26990,41760,63900,True,NaN


### Rough Wrangle State Field Descriptions

In [38]:
sta_desc = sta.parse('Field Descriptions')

In [39]:
sta_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [40]:
sta_desc = sta_desc.drop('Unnamed: 2', axis=1)
drop_index = sta_desc.index[0:8]
sta_desc_dropped = sta_desc.drop(drop_index)
sta_desc_new_header = sta_desc_dropped.iloc[0]
sta_desc_new = sta_desc_dropped[1:]
sta_desc_new.columns = sta_desc_new_header
sta_desc_new = sta_desc_new.reset_index(drop=True)
sta_desc_new.index.name = None
sta_desc_new = sta_desc_new.rename_axis(None, axis=1)

In [41]:
sta_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


## Metropolitan Data

### Rough Wrangle Metropolitan Data

In [42]:
met_data = met.parse('MSA_M2024_dl')
met_data.columns = met_data.columns.str.lower()

In [43]:
met_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,20.25,29.34,45.3,23120,29640,42120,61020,94220,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,42.03,61.17,84.3,44710,61250,87420,127230,175350,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,85.35,#,#,99140,107040,177520,#,#,NaN,NaN
3,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,37.18,58.64,87.22,35980,50640,77340,121970,181420,NaN,NaN
4,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,...,50.02,73.86,99.9,58480,82560,104030,153630,207790,NaN,NaN


### Rough Wrangle Metropolitan Field Descriptions

In [44]:
met_desc = met.parse('Field Descriptions')

In [45]:
met_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [46]:
met_desc = met_desc.drop('Unnamed: 2', axis=1)
drop_index = met_desc.index[0:8]
met_desc_dropped = met_desc.drop(drop_index)
met_desc_new_header = met_desc_dropped.iloc[0]
met_desc_new = met_desc_dropped[1:]
met_desc_new.columns = met_desc_new_header
met_desc_new = met_desc_new.reset_index(drop=True)
met_desc_new.index.name = None
met_desc_new = met_desc_new.rename_axis(None, axis=1)

In [47]:
met_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


## Balance of State Data

### Rough Wrangle Balance of State Data

In [48]:
bos_data = bos.parse('BOS_M2024_dl')
bos_data.columns = bos_data.columns.str.lower() 

In [49]:
bos_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,18.94,25,35.51,22380,29690,39400,52000,73870,NaN,NaN
1,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,40.54,58.5,80.72,44770,60730,84320,121680,167910,NaN,NaN
2,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,71.58,85.16,#,114610,121930,148890,177130,#,NaN,NaN
3,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,46.63,73.99,101.78,42680,63380,96980,153900,211700,NaN,NaN
4,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18210,22690,51460,52020,52020,True,NaN


### Rough Wrangle Balance of State Field Descriptions

In [50]:
bos_desc = bos.parse('Field Descriptions')

In [51]:
bos_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [52]:
bos_desc = bos_desc.drop('Unnamed: 2', axis=1)
drop_index = bos_desc.index[0:8]
bos_desc_dropped = bos_desc.drop(drop_index)
bos_desc_new_header = bos_desc_dropped.iloc[0]
bos_desc_new = bos_desc_dropped[1:]
bos_desc_new.columns = bos_desc_new_header
bos_desc_new = bos_desc_new.reset_index(drop=True)
bos_desc_new.index.name = None
bos_desc_new = bos_desc_new.rename_axis(None, axis=1)

In [53]:
bos_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


## National Sector Data

### Rough Wrangle National Sector Data

In [54]:
nbs_data = nbs.parse('Natsector_M2024_dl')
nbs_data.columns = nbs_data.columns.str.lower() 

In [55]:
nbs_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,00-0000,All Occupations,...,17.66,22.21,30.23,33280,34680,36720,46200,62880,NaN,NaN
1,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-0000,Management Occupations,...,46.08,63.4,90.74,41710,67090,95840,131870,188730,NaN,NaN
2,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1000,Top Executives,...,40.91,61.6,91.98,32100,57390,85080,128140,191320,NaN,NaN
3,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1010,Chief Executives,...,103.99,#,#,105020,154280,216290,#,#,NaN,NaN
4,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1011,Chief Executives,...,103.99,#,#,105020,154280,216290,#,#,NaN,NaN


### Rough Wrangle National Sector Field Descriptions

In [56]:
nbs_desc = nbs.parse('Field Descriptions')

In [57]:
nbs_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [58]:
nbs_desc = nbs_desc.drop('Unnamed: 2', axis=1)
drop_index = nbs_desc.index[0:8]
nbs_desc_dropped = nbs_desc.drop(drop_index)
nbs_desc_new_header = nbs_desc_dropped.iloc[0]
nbs_desc_new = nbs_desc_dropped[1:]
nbs_desc_new.columns = nbs_desc_new_header
nbs_desc_new = nbs_desc_new.reset_index(drop=True)
nbs_desc_new.index.name = None
nbs_desc_new = nbs_desc_new.rename_axis(None, axis=1)

In [59]:
nbs_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


## Rough Wrangle Closing Commentary

The `Field Description` dataframes are ready to be pushed to the sql database. I do want them readily available as a part of the long-term project, so that either I or my users can reference them.

The data for each of the five sets, on the other hand, is a mixed bag. I'm seeing `NaN` values and `#` values. I do not know what they mean. It's time to start investigating there.

# Full Wrangle

## National Data

### Basic Inspection

In [60]:
nat_data.dtypes

area              int64
area_title       object
area_type         int64
prim_state       object
naics             int64
naics_title      object
i_group          object
own_code          int64
occ_code         object
occ_title        object
o_group          object
tot_emp           int64
emp_prse        float64
jobs_1000       float64
loc_quotient    float64
pct_total       float64
pct_rpt         float64
h_mean           object
a_mean           object
mean_prse       float64
h_pct10          object
h_pct25          object
h_median         object
h_pct75          object
h_pct90          object
a_pct10          object
a_pct25          object
a_median         object
a_pct75          object
a_pct90          object
annual           object
hourly           object
dtype: object

In [61]:
nat_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1403 entries, 0 to 1402
Data columns (total 32 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area          1403 non-null   int64  
 1   area_title    1403 non-null   object 
 2   area_type     1403 non-null   int64  
 3   prim_state    1403 non-null   object 
 4   naics         1403 non-null   int64  
 5   naics_title   1403 non-null   object 
 6   i_group       1403 non-null   object 
 7   own_code      1403 non-null   int64  
 8   occ_code      1403 non-null   object 
 9   occ_title     1403 non-null   object 
 10  o_group       1403 non-null   object 
 11  tot_emp       1403 non-null   int64  
 12  emp_prse      1403 non-null   float64
 13  jobs_1000     0 non-null      float64
 14  loc_quotient  0 non-null      float64
 15  pct_total     0 non-null      float64
 16  pct_rpt       0 non-null      float64
 17  h_mean        1403 non-null   object 
 18  a_mean        1403 non-null 

### Removing Blank Columns

We see that there are four entirely blank columns.

Let's check their description.

In [62]:
with pd.option_context('display.max_colwidth', None):
    print(nat_desc.iloc[13:17,:])

           Field  \
13     jobs_1000   
14  loc quotient   
15     pct_total   
16       pct_rpt   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      Field Description  
13                                                                                                                                                                                                                                                                                                                          The number of jobs (employment) in the given occupation per 1,000 jobs in the given area. Onl

#### Confirm Necessity of Deletion

These four columns can be dropped from the `nat_data` df. 

#### Add To Do List

##### Data
- Drop the four columns that have all blanks

##### Descriptions
- Drop the rows to match the dropped columns in the data

##### Remote Columns from National Data

In [63]:
drop_nat_data_list = ['jobs_1000', 'loc_quotient', 'pct_total', 'pct_rpt']

In [64]:
nat_data = nat_data.drop(drop_nat_data_list, axis=1)

In [65]:
nat_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1403 entries, 0 to 1402
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1403 non-null   int64  
 1   area_title   1403 non-null   object 
 2   area_type    1403 non-null   int64  
 3   prim_state   1403 non-null   object 
 4   naics        1403 non-null   int64  
 5   naics_title  1403 non-null   object 
 6   i_group      1403 non-null   object 
 7   own_code     1403 non-null   int64  
 8   occ_code     1403 non-null   object 
 9   occ_title    1403 non-null   object 
 10  o_group      1403 non-null   object 
 11  tot_emp      1403 non-null   int64  
 12  emp_prse     1403 non-null   float64
 13  h_mean       1403 non-null   object 
 14  a_mean       1403 non-null   object 
 15  mean_prse    1403 non-null   float64
 16  h_pct10      1403 non-null   object 
 17  h_pct25      1403 non-null   object 
 18  h_median     1403 non-null   object 
 19  h_pct7

##### Remote Rows from National Descriptions

In [66]:
nat_desc = nat_desc[nat_desc['Field'].isin(drop_nat_data_list) == False]

In [67]:
nat_desc

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...
5,naics_title,North American Industry Classification System ...
6,i_group,Industry level. Indicates cross-industry or NA...
7,own_code,Ownership type: 1= Federal Government; 2= Stat...
8,occ_code,The 6-digit Standard Occupational Classificati...
9,occ_title,SOC title or OEWS-specific title for the occup...


#### Cross Off To Do List

##### Data
- ~~Drop the four columns that have all blanks~~

##### Descriptions
- ~~Drop the rows to match the dropped columns in the data~~

### National Data Area-Related Columns Wrangling

Dealing with the first four rows, `area`, `area_title`, `area_type`, `prim_state`.

The first and third row are of type `int`, while the second and fourths are`object` types, suggesting either strings or even multiple types.

All values are non-null.

We need to make sure that the first and third do not contain values outside the scope outline in the description.

We need to inspect the second and fourth for consistency as well.

In [68]:
nat_data['area'].describe()

count    1403.0
mean       99.0
std         0.0
min        99.0
25%        99.0
50%        99.0
75%        99.0
max        99.0
Name: area, dtype: float64

That's interesting. Apparently, the entire first row is essentially meaningless.

In [69]:
print(nat_desc.loc[nat_desc['Field'] == 'area', 'Field Description'].values[0])

U.S. (99), state FIPS code, Metropolitan Statistical Area (MSA) code, or OEWS-specific nonmetropolitan area code 


This appears to be a column that only has variance in other datasets. The value of this column in this context is limited, but it may be useful elsewhere. Therefore, we will keep it as is, but perhaps discard it when we begin inference.

In [70]:
nat_data['area_title'].head()

0    U.S.
1    U.S.
2    U.S.
3    U.S.
4    U.S.
Name: area_title, dtype: object

In [71]:
nat_data['area_title'].unique()

array(['U.S.'], dtype=object)

The lack of variance in the `area_title` column follows.

In [72]:
nat_data['area_type'].unique()

array([1])

The results for `area_type` are identical.

In [73]:
nat_data['prim_state'].unique()

array(['US'], dtype=object)

The result of this study is that these columns all simply indicate that they are exactly what the title of the dataset would imply: National.

We will keep these for the PostgreSQL insertion at this time.

### Industry Classifications

Looking at `naics`, `naics_title`, and `i_group`.

In [74]:
nat_data['naics'].unique()

array([0])

In [75]:
print(nat_desc.loc[nat_desc['Field'] == 'naics', 'Field Description'].values[0])

North American Industry Classification System (NAICS) code for the given industry 


In [76]:
nat_data['naics_title'].unique()

array(['Cross-industry'], dtype=object)

In [77]:
print(nat_desc.loc[nat_desc['Field'] == 'naics_title', 'Field Description'].values[0])

North American Industry Classification System (NAICS) title for the given industry 


In [78]:
nat_data['i_group'].unique()

array(['cross-industry'], dtype=object)

In [79]:
print(nat_desc.loc[nat_desc['Field'] == 'i_group', 'Field Description'].values[0])

Industry level. Indicates cross-industry or NAICS sector, 3-digit, 4-digit, 5-digit, or 6-digit industry. For industries that OEWS no longer publishes at the 4-digit NAICS level, the “4-digit” designation indicates the most detailed industry breakdown available: either a standard NAICS 3-digit industry or an OEWS-specific combination of 4-digit industries. Industries that OEWS has aggregated to the 3-digit NAICS level (for example, NAICS 327000) will appear twice, once with the “3-digit” and once with the “4-digit” designation. 


In [80]:
nat_data['own_code'].unique()	

array([1235])

In [81]:
print(nat_desc.loc[nat_desc['Field'] == 'own_code', 'Field Description'].values[0])

Ownership type: 1= Federal Government; 2= State Government; 3= Local Government; 123= Federal, State, and Local Government; 235=Private, State, and Local Government; 35 = Private and Local Government; 5= Private; 57=Private, Local Government Gambling Establishments (Sector 71), and Local Government Casino Hotels (Sector 72); 58= Private plus State and Local Government Hospitals; 59= Private and Postal Service; 1235= Federal, State, and Local Government and Private Sector 


These columns follow the same pattern as before in terms of variance.

We will keep the values for the PostgreSQL database, even though they will not have value in isolation from other datasets.

### `occ_code`: Standard Occupation Classifications

In [82]:
nat_data['occ_code'].unique()

array(['00-0000', '11-0000', '11-1000', ..., '53-7121', '53-7190',
       '53-7199'], dtype=object)

In [83]:
print(nat_desc.loc[nat_desc['Field'] == 'occ_code', 'Field Description'].values[0])

The 6-digit Standard Occupational Classification (SOC) code or OEWS-specific code for the occupation 


In [84]:
nat_data['occ_code'].describe()

count        1403
unique       1396
top       31-1120
freq            2
Name: occ_code, dtype: object

In [85]:
nat_data['occ_code'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 1403 entries, 0 to 1402
Series name: occ_code
Non-Null Count  Dtype 
--------------  ----- 
1403 non-null   object
dtypes: object(1)
memory usage: 11.1+ KB


In [86]:
print(type(nat_data['occ_code'][0]))

<class 'str'>


In [87]:
counts = nat_data['occ_code'].value_counts()

In [88]:
counts

occ_code
31-1120    2
47-4090    2
29-2010    2
51-2090    2
39-7010    2
          ..
27-2012    1
27-2011    1
27-2010    1
27-2000    1
53-7199    1
Name: count, Length: 1396, dtype: int64

In [89]:
threshold = 1

In [90]:
counts_thresh = counts[counts > threshold].index

In [91]:
counts_thresh

Index(['31-1120', '47-4090', '29-2010', '51-2090', '39-7010', '13-1020',
       '13-2020'],
      dtype='object', name='occ_code')

It seems that there are a few areas that are duplicated.

In [92]:
nat_data[nat_data.groupby('occ_code')['occ_code'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
78,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-1020,Buyers and Purchasing Agents,...,36.37,47.69,61.31,46460,58670,75650,99190,127520,NaN,NaN
79,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-1020,Buyers and Purchasing Agents,...,36.37,47.69,61.31,46460,58670,75650,99190,127520,NaN,NaN
111,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-2020,Property Appraisers and Assessors,...,31.45,43.66,59.02,38480,49310,65420,90810,122760,NaN,NaN
112,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-2020,Property Appraisers and Assessors,...,31.45,43.66,59.02,38480,49310,65420,90810,122760,NaN,NaN
573,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-2010,Clinical Laboratory Technologists and Technicians,...,29.75,38.46,47.11,38020,46580,61890,80010,97990,NaN,NaN
574,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-2010,Clinical Laboratory Technologists and Technicians,...,29.75,38.46,47.11,38020,46580,61890,80010,97990,NaN,NaN
612,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,31-1120,Home Health and Personal Care Aides,...,16.78,18.26,21.25,25600,30370,34900,37980,44190,NaN,NaN
613,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,31-1120,Home Health and Personal Care Aides,...,16.78,18.26,21.25,25600,30370,34900,37980,44190,NaN,NaN
779,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN
780,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN


We appear to have found fourteen duplicate rows. These must be cleaned.

#### Add To Do Item

- Clean data duplicates

Before cleaning them, let's take a moment to understand by looking at the [SOC User Guide](https://www.bls.gov/soc/socguide.htm#Ques3).

> The 2000 SOC classifies workers at four levels of aggregation: 1) major group; 2) minor group; 3) broad occupation; and 4) detailed occupation. All occupations are clustered into one of the following 23 major groups:

>  Within these major groups are 96 minor groups, 449 broad occupations, and 821 detailed occupations. Occupations with similar skills or work activities are grouped at each of the four levels of hierarchy to facilitate comparisons. For example, "Life, Physical and Social Science Occupations" (19-0000) is divided into four minor groups, "Life Scientists" (19-1000), "Physical Scientists" (19-2000), "Social Scientists and Related Workers" (19-3000), and "Life, Physical and Social Science Technicians" (19-4000). Life Scientists contains broad occupations such as "Agriculture and Food Scientists" (19-1010), and "Biological Scientists" (19-1020). The broad occupation Biological Scientists includes detailed occupations such as "Biochemists and Biophysicists" (19-1021), and "Microbiologists" (19-1022).

How many `o_group` values do we have before we begin cleaning?

In [93]:
nat_data['o_group'].value_counts()

o_group
detailed    831
broad       455
minor        94
major        22
total         1
Name: count, dtype: int64

We appear to have less of each group than the number reported in the user guide. 

Since this dataset has not yet dropped any rows, we can assume that the dataset originated in this form.

#### Clean Duplicate Rows

In [94]:
nat_data = nat_data.drop_duplicates()

In [95]:
counts = nat_data['occ_code'].value_counts()

In [96]:
counts

occ_code
31-1120    2
47-4090    2
29-2010    2
51-2090    2
39-7010    2
          ..
27-2012    1
27-2011    1
27-2010    1
27-2000    1
53-7199    1
Name: count, Length: 1396, dtype: int64

They don't seem to be dropped. Perhaps there are hidden rows that are different?

In [97]:
with pd.option_context('display.max_columns', None):
    print(nat_data[nat_data.groupby('occ_code')['occ_code'].transform('count') > threshold])

      area area_title  area_type prim_state  naics     naics_title  \
78      99       U.S.          1         US      0  Cross-industry   
79      99       U.S.          1         US      0  Cross-industry   
111     99       U.S.          1         US      0  Cross-industry   
112     99       U.S.          1         US      0  Cross-industry   
573     99       U.S.          1         US      0  Cross-industry   
574     99       U.S.          1         US      0  Cross-industry   
612     99       U.S.          1         US      0  Cross-industry   
613     99       U.S.          1         US      0  Cross-industry   
779     99       U.S.          1         US      0  Cross-industry   
780     99       U.S.          1         US      0  Cross-industry   
1045    99       U.S.          1         US      0  Cross-industry   
1046    99       U.S.          1         US      0  Cross-industry   
1163    99       U.S.          1         US      0  Cross-industry   
1164    99       U.S

This shows us that the `o_group` column is providing some type of difference in these columns. Let's explore this column's description.

In [98]:
print(nat_desc.loc[nat_desc['Field'] == 'o_group', 'Field Description'].values[0])

SOC occupation level. For most occupations, this field indicates the standard SOC major, minor, broad, and detailed levels, in addition to all-occupations totals. For occupations that OEWS no longer publishes at the SOC detailed level, the “detailed” designation indicates the most detailed data available: either a standard SOC broad occupation or an OEWS-specific combination of detailed occupations. Occupations that OEWS has aggregated to the SOC broad occupation level will appear in the file twice, once with the “broad” and once with the “detailed” designation.


This informs us that these are rows of data that have been included on purpose twice, due to changes in the way Occupational Employment and Wage Statistics (OEWS) manages the data.

The duplicates that have the `detailed` values can be dropped, as OEWS has aggregated them into the `broad` category.

In [99]:
nat_data_filtered = nat_data[(nat_data['occ_code'].isin(counts_thresh)) & (nat_data['o_group'] == 'detailed')]

In [100]:
nat_data_filtered

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
79,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-1020,Buyers and Purchasing Agents,...,36.37,47.69,61.31,46460,58670,75650,99190,127520,NaN,NaN
112,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-2020,Property Appraisers and Assessors,...,31.45,43.66,59.02,38480,49310,65420,90810,122760,NaN,NaN
574,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-2010,Clinical Laboratory Technologists and Technicians,...,29.75,38.46,47.11,38020,46580,61890,80010,97990,NaN,NaN
613,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,31-1120,Home Health and Personal Care Aides,...,16.78,18.26,21.25,25600,30370,34900,37980,44190,NaN,NaN
780,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN
1046,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,47-4090,Miscellaneous Construction and Related Workers,...,23.14,29.31,37.28,35160,39990,48120,60960,77540,NaN,NaN
1164,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,51-2090,Miscellaneous Assemblers and Fabricators,...,20.29,23.76,29.82,31650,36660,42210,49410,62030,NaN,NaN


In [101]:
nat_data_filtered['o_group']

79      detailed
112     detailed
574     detailed
613     detailed
780     detailed
1046    detailed
1164    detailed
Name: o_group, dtype: object

In [102]:
nat_data_pre_drop = nat_data.drop(nat_data_filtered.index)

In [103]:
nat_data_pre_drop.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN


In [104]:
nat_data_pre_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1396 entries, 0 to 1402
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1396 non-null   int64  
 1   area_title   1396 non-null   object 
 2   area_type    1396 non-null   int64  
 3   prim_state   1396 non-null   object 
 4   naics        1396 non-null   int64  
 5   naics_title  1396 non-null   object 
 6   i_group      1396 non-null   object 
 7   own_code     1396 non-null   int64  
 8   occ_code     1396 non-null   object 
 9   occ_title    1396 non-null   object 
 10  o_group      1396 non-null   object 
 11  tot_emp      1396 non-null   int64  
 12  emp_prse     1396 non-null   float64
 13  h_mean       1396 non-null   object 
 14  a_mean       1396 non-null   object 
 15  mean_prse    1396 non-null   float64
 16  h_pct10      1396 non-null   object 
 17  h_pct25      1396 non-null   object 
 18  h_median     1396 non-null   object 
 19  h_pct75    

We see that seven rows have been removed.

In [105]:
nat_data_pre_drop[nat_data_pre_drop.groupby('occ_code')['occ_code'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly


There are no duplicates.

In [106]:
nat_data = nat_data_pre_drop

#### Mark To Do Item Complete

- ~~Clean data duplicates~~

### `occ_title`: SOC Column Cleaning Continuation

In [107]:
nat_data['occ_title'].unique()

array(['All Occupations', 'Management Occupations', 'Top Executives', ...,
       'Tank Car, Truck, and Ship Loaders',
       'Miscellaneous Material Moving Workers',
       'Material Moving Workers, All Other'], dtype=object)

In [108]:
print(nat_desc.loc[nat_desc['Field'] == 'occ_title', 'Field Description'].values[0])

SOC title or OEWS-specific title for the occupation


In [109]:
nat_data['occ_title'].describe()

count                   1396
unique                  1138
top       Massage Therapists
freq                       2
Name: occ_title, dtype: object

In [110]:
nat_data['occ_title'].info()

<class 'pandas.core.series.Series'>
Index: 1396 entries, 0 to 1402
Series name: occ_title
Non-Null Count  Dtype 
--------------  ----- 
1396 non-null   object
dtypes: object(1)
memory usage: 21.8+ KB


In [111]:
print(type(nat_data['occ_title'][0]))

<class 'str'>


In [112]:
counts = nat_data['occ_title'].value_counts()

In [113]:
counts

occ_title
Massage Therapists                                 2
Biological Technicians                             2
Couriers and Messengers                            2
Cargo and Freight Agents                           2
Printing Workers                                   2
                                                  ..
Healthcare Diagnosing or Treating Practitioners    1
Dentists                                           1
Dentists, General                                  1
Oral and Maxillofacial Surgeons                    1
Material Moving Workers, All Other                 1
Name: count, Length: 1138, dtype: int64

We seem to have found more duplicate values.

In [114]:
nat_data[nat_data.groupby('occ_title')['occ_title'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
5,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1020,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
6,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
7,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1030,Legislators,...,*,*,*,20380,29120,44810,80350,137820,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1386,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7051,Industrial Truck and Tractor Operators,...,22.3,25.81,29.59,36500,39780,46390,53680,61540,NaN,NaN
1397,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7080,Refuse and Recyclable Material Collectors,...,23.24,29.33,36.16,31810,38330,48350,61010,75200,NaN,NaN
1398,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7081,Refuse and Recyclable Material Collectors,...,23.24,29.33,36.16,31810,38330,48350,61010,75200,NaN,NaN
1399,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7120,"Tank Car, Truck, and Ship Loaders",...,27.92,34.25,42.36,38260,47260,58070,71230,88120,NaN,NaN


In [115]:
with pd.option_context('display.max_columns', None):
    print(nat_data[nat_data.groupby('occ_title')['occ_title'].transform('count') > threshold].head(2))

   area area_title  area_type prim_state  naics     naics_title  \
3    99       U.S.          1         US      0  Cross-industry   
4    99       U.S.          1         US      0  Cross-industry   

          i_group  own_code occ_code         occ_title   o_group  tot_emp  \
3  cross-industry      1235  11-1010  Chief Executives     broad   211850   
4  cross-industry      1235  11-1011  Chief Executives  detailed   211850   

   emp_prse  h_mean  a_mean  mean_prse h_pct10 h_pct25 h_median h_pct75  \
3       1.2  126.41  262930        0.9   35.44   60.61    99.24       #   
4       1.2  126.41  262930        0.9   35.44   60.61    99.24       #   

  h_pct90 a_pct10 a_pct25 a_median a_pct75 a_pct90 annual hourly  
3       #   73710  126080   206420       #       #    NaN    NaN  
4       #   73710  126080   206420       #       #    NaN    NaN  


These duplicates seem to have a different value in the `occ_code` and the `o_group` code. 

Likely, the `o_group` issue is a continuation of the issue from before, where a formerly `detailed` dataset has been aggregated into a `broad` dataset.

The `occ_code` appears to be raised by one digit lower as a part of the aggregation methodology shift.

Let's consider that the `detailed` rows can, again, are duplicates and can therefore be dropped. 

#### Add To Do Item

- Remove duplicate rows discovered via the `occ_title` column.

#### Clean Duplicate Rows

In [116]:
counts_thresh = counts[counts > threshold].index

In [117]:
nat_data_filtered = nat_data[(nat_data['occ_title'].isin(counts_thresh)) & (nat_data['o_group'] == 'detailed')]

In [118]:
nat_data_filtered

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
6,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
8,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,20380,29120,44810,80350,137820,True,NaN
11,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-2011,Advertising and Promotions Managers,...,61.04,85.85,#,63000,85990,126960,178570,#,NaN,NaN
23,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-3021,Computer and Information Systems Managers,...,82.31,103.95,#,104450,134350,171200,216220,#,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1382,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7031,Dredge Operators,...,23.28,28.99,36.08,42060,46120,48430,60300,75050,NaN,NaN
1384,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7041,Hoist and Winch Operators,...,25.15,43.37,55.83,33910,39220,52310,90200,116120,NaN,NaN
1386,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7051,Industrial Truck and Tractor Operators,...,22.3,25.81,29.59,36500,39780,46390,53680,61540,NaN,NaN
1398,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7081,Refuse and Recyclable Material Collectors,...,23.24,29.33,36.16,31810,38330,48350,61010,75200,NaN,NaN


In [119]:
nat_data_filtered['o_group'].unique()

array(['detailed'], dtype=object)

On the one hand, we seem to have made some progress. We have found `249` rows that can be dropped. 

However, there were `516` rows originally. We are missing perhaps `9` rows. 

While we could write some complicated logic at this stage to try to figure out why those rows are not yet included, we don't have to do so. When we drop the known duplicate rows and rerun the original duplicate-finding logic, we'll see them in isolation.

In [120]:
nat_data_pre_drop = nat_data.drop(nat_data_filtered.index)

In [121]:
nat_data_pre_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1147 entries, 0 to 1402
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1147 non-null   int64  
 1   area_title   1147 non-null   object 
 2   area_type    1147 non-null   int64  
 3   prim_state   1147 non-null   object 
 4   naics        1147 non-null   int64  
 5   naics_title  1147 non-null   object 
 6   i_group      1147 non-null   object 
 7   own_code     1147 non-null   int64  
 8   occ_code     1147 non-null   object 
 9   occ_title    1147 non-null   object 
 10  o_group      1147 non-null   object 
 11  tot_emp      1147 non-null   int64  
 12  emp_prse     1147 non-null   float64
 13  h_mean       1147 non-null   object 
 14  a_mean       1147 non-null   object 
 15  mean_prse    1147 non-null   float64
 16  h_pct10      1147 non-null   object 
 17  h_pct25      1147 non-null   object 
 18  h_median     1147 non-null   object 
 19  h_pct75    

#### Part 2: Analyzing for Additional Duplicates via `occ_title`

With the first set of duplicates removed, we should now be able to see the remaining duplicates.

In [122]:
counts = nat_data_pre_drop['occ_title'].value_counts()

In [123]:
counts

occ_title
Supervisors of Food Preparation and Serving Workers    2
Baggage Porters, Bellhops, and Concierges              2
Sales Representatives, Wholesale and Manufacturing     2
Grounds Maintenance Workers                            2
Tour and Travel Guides                                 2
                                                      ..
Athletes, Coaches, Umpires, and Related Workers        1
Athletes and Sports Competitors                        1
Coaches and Scouts                                     1
Umpires, Referees, and Other Sports Officials          1
Material Moving Workers, All Other                     1
Name: count, Length: 1138, dtype: int64

In [124]:
nat_data_pre_drop[nat_data_pre_drop.groupby('occ_title')['occ_title'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
304,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,19-5000,Occupational Health and Safety Specialists and...,...,37.93,48.79,61.13,47790,59620,78900,101490,127140,NaN,NaN
305,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,19-5010,Occupational Health and Safety Specialists and...,...,37.93,48.79,61.13,47790,59620,78900,101490,127140,NaN,NaN
681,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,35-1000,Supervisors of Food Preparation and Serving Wo...,...,21.22,27.4,34.58,29600,35840,44140,56990,71920,NaN,NaN
682,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,35-1010,Supervisors of Food Preparation and Serving Wo...,...,21.22,27.4,34.58,29600,35840,44140,56990,71920,NaN,NaN
725,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,37-3000,Grounds Maintenance Workers,...,18.5,22.3,27.14,30140,35470,38470,46390,56460,NaN,NaN
726,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,37-3010,Grounds Maintenance Workers,...,18.5,22.3,27.14,30140,35470,38470,46390,56460,NaN,NaN
774,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-6000,"Baggage Porters, Bellhops, and Concierges",...,17.7,20.96,26.13,28360,32560,36810,43600,54340,NaN,NaN
775,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-6010,"Baggage Porters, Bellhops, and Concierges",...,17.7,20.96,26.13,28360,32560,36810,43600,54340,NaN,NaN
778,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7000,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN
779,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN


There are still `9` duplicates, as expected.

In [125]:
with pd.option_context('display.max_columns', None):
    print(nat_data_pre_drop[nat_data_pre_drop.groupby('occ_title')['occ_title'].transform('count') > threshold])

      area area_title  area_type prim_state  naics     naics_title  \
304     99       U.S.          1         US      0  Cross-industry   
305     99       U.S.          1         US      0  Cross-industry   
681     99       U.S.          1         US      0  Cross-industry   
682     99       U.S.          1         US      0  Cross-industry   
725     99       U.S.          1         US      0  Cross-industry   
726     99       U.S.          1         US      0  Cross-industry   
774     99       U.S.          1         US      0  Cross-industry   
775     99       U.S.          1         US      0  Cross-industry   
778     99       U.S.          1         US      0  Cross-industry   
779     99       U.S.          1         US      0  Cross-industry   
816     99       U.S.          1         US      0  Cross-industry   
817     99       U.S.          1         US      0  Cross-industry   
917     99       U.S.          1         US      0  Cross-industry   
918     99       U.S

This shows that these duplicates are due to a difference between `minor` and `broad` specifications in the `o_group` column. 

In [126]:
print(nat_desc.loc[nat_desc['Field'] == 'o_group', 'Field Description'].values[0])

SOC occupation level. For most occupations, this field indicates the standard SOC major, minor, broad, and detailed levels, in addition to all-occupations totals. For occupations that OEWS no longer publishes at the SOC detailed level, the “detailed” designation indicates the most detailed data available: either a standard SOC broad occupation or an OEWS-specific combination of detailed occupations. Occupations that OEWS has aggregated to the SOC broad occupation level will appear in the file twice, once with the “broad” and once with the “detailed” designation.


The description of this column does not provide information about how to handle these `minor`/`broad` duplicates. 

They are clearly duplicates, as evidenced by the identical data in nearly all other columns between the matching pairs.

Therefore, at least one duplicate for each needs to be dropped. 

Let's revisit the [SOC User Guide](https://www.bls.gov/soc/socguide.htm):

> The 2000 SOC classifies workers at four levels of aggregation: 1) major group; 2) minor group; 3) broad occupation; and 4) detailed occupation. All occupations are clustered into one of the following 23 major groups:

>  Within these major groups are 96 minor groups, 449 broad occupations, and 821 detailed occupations. Occupations with similar skills or work activities are grouped at each of the four levels of hierarchy to facilitate comparisons. For example, "Life, Physical and Social Science Occupations" (19-0000) is divided into four minor groups, "Life Scientists" (19-1000), "Physical Scientists" (19-2000), "Social Scientists and Related Workers" (19-3000), and "Life, Physical and Social Science Technicians" (19-4000). Life Scientists contains broad occupations such as "Agriculture and Food Scientists" (19-1010), and "Biological Scientists" (19-1020). The broad occupation Biological Scientists includes detailed occupations such as "Biochemists and Biophysicists" (19-1021), and "Microbiologists" (19-1022).

This creates the vague understanding that these row duplicates are the same occupational fields surveyed, but the fields are categorized twice -- as both `minor` and `broad` groups. The purpose for this duplication is unknown.

According to the user guide, the `minor` group only has `96` instances across the whole database. We already have only `94` such groups, as shown previously.

We do know that when BLS adds duplicates, it does so because it has aggregated data from a more `detailed` into a more `broad` direction. The BLS has a track record of making things more general, not more detailed.

Let's continue in this direction by dropping the `broad` rows and keeping the more general `minor` rows.

#### Part 3: Dropping Additional Duplicates

In [127]:
counts_thresh = counts[counts > threshold].index

In [128]:
nat_data_filtered = nat_data_pre_drop[(nat_data_pre_drop['occ_title'].isin(counts_thresh)) & (nat_data_pre_drop['o_group'] == 'broad')]

In [129]:
nat_data_filtered

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
305,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,19-5010,Occupational Health and Safety Specialists and...,...,37.93,48.79,61.13,47790,59620,78900,101490,127140,NaN,NaN
682,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,35-1010,Supervisors of Food Preparation and Serving Wo...,...,21.22,27.4,34.58,29600,35840,44140,56990,71920,NaN,NaN
726,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,37-3010,Grounds Maintenance Workers,...,18.5,22.3,27.14,30140,35470,38470,46390,56460,NaN,NaN
775,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-6010,"Baggage Porters, Bellhops, and Concierges",...,17.7,20.96,26.13,28360,32560,36810,43600,54340,NaN,NaN
779,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN
817,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,41-4010,"Sales Representatives, Wholesale and Manufactu...",...,35.63,49.61,74.97,38910,50820,74100,103200,155930,NaN,NaN
918,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,43-6010,Secretaries and Administrative Assistants,...,22.82,28.84,36.81,33840,38790,47460,59990,76550,NaN,NaN
1022,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,47-3010,"Helpers, Construction Trades",...,19.44,22.94,27.64,31360,36280,40430,47710,57480,NaN,NaN
1213,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,51-5110,Printing Workers,...,21.55,25.24,30.06,31450,36950,44830,52510,62530,NaN,NaN


In [130]:
nat_data_pre_drop_2 = nat_data_pre_drop.drop(nat_data_filtered.index)

In [131]:
nat_data_pre_drop_2

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
5,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1020,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1396,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7073,Wellhead Pumpers,...,33.66,38.81,46.86,39110,54450,70010,80720,97470,NaN,NaN
1397,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7080,Refuse and Recyclable Material Collectors,...,23.24,29.33,36.16,31810,38330,48350,61010,75200,NaN,NaN
1399,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7120,"Tank Car, Truck, and Ship Loaders",...,27.92,34.25,42.36,38260,47260,58070,71230,88120,NaN,NaN
1401,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7190,Miscellaneous Material Moving Workers,...,20.04,24.45,31.18,33280,35470,41690,50850,64850,NaN,NaN


Before this data cleaning instance, we had `1396` rows. Our original number of rows that had duplicate data was `516`.

In [132]:
(1396 - 1138) * 2

516

We appear to have dropped the correct amount of rows for this attempt.

In [133]:
nat_data_pre_drop_2[nat_data_pre_drop_2.groupby('occ_title')['occ_title'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly


In [134]:
nat_data_pre_drop_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1138 entries, 0 to 1402
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1138 non-null   int64  
 1   area_title   1138 non-null   object 
 2   area_type    1138 non-null   int64  
 3   prim_state   1138 non-null   object 
 4   naics        1138 non-null   int64  
 5   naics_title  1138 non-null   object 
 6   i_group      1138 non-null   object 
 7   own_code     1138 non-null   int64  
 8   occ_code     1138 non-null   object 
 9   occ_title    1138 non-null   object 
 10  o_group      1138 non-null   object 
 11  tot_emp      1138 non-null   int64  
 12  emp_prse     1138 non-null   float64
 13  h_mean       1138 non-null   object 
 14  a_mean       1138 non-null   object 
 15  mean_prse    1138 non-null   float64
 16  h_pct10      1138 non-null   object 
 17  h_pct25      1138 non-null   object 
 18  h_median     1138 non-null   object 
 19  h_pct75    

In [135]:
nat_data = nat_data_pre_drop_2

#### Remove To Do Item

- ~~Remove duplicate rows discovered via the `occ_title` column.~~

### o_group: SOC Column Cleaning Continued

In [136]:
nat_data['o_group'].unique()

array(['total', 'major', 'minor', 'broad', 'detailed'], dtype=object)

In [137]:
nat_data['o_group'].value_counts()

o_group
detailed    575
broad       446
minor        94
major        22
total         1
Name: count, dtype: int64

The `total` value in the `o_group` column is new to this analysis.

In [138]:
print(nat_desc.loc[nat_desc['Field'] == 'o_group', 'Field Description'].values[0])

SOC occupation level. For most occupations, this field indicates the standard SOC major, minor, broad, and detailed levels, in addition to all-occupations totals. For occupations that OEWS no longer publishes at the SOC detailed level, the “detailed” designation indicates the most detailed data available: either a standard SOC broad occupation or an OEWS-specific combination of detailed occupations. Occupations that OEWS has aggregated to the SOC broad occupation level will appear in the file twice, once with the “broad” and once with the “detailed” designation.


We seem to have completed the process of cleaning this column at this time. However, as we delve deeper into the data, perhaps more revelations may shed light on data quality.

### 'tot_emp' Wrangle

In [139]:
nat_data['tot_emp'].duplicated().sum()

np.int64(68)

We seem to have `68` rows with duplicated values in `tot_emp`.

In [140]:
with pd.option_context('display.max_columns', None):
    print(nat_data[nat_data.groupby('tot_emp')['tot_emp'].transform('count') > threshold].sort_values(by='tot_emp', ascending=False))

      area area_title  area_type prim_state  naics     naics_title  \
835     99       U.S.          1         US      0  Cross-industry   
836     99       U.S.          1         US      0  Cross-industry   
793     99       U.S.          1         US      0  Cross-industry   
792     99       U.S.          1         US      0  Cross-industry   
815     99       U.S.          1         US      0  Cross-industry   
...    ...        ...        ...        ...    ...             ...   
1005    99       U.S.          1         US      0  Cross-industry   
844     99       U.S.          1         US      0  Cross-industry   
843     99       U.S.          1         US      0  Cross-industry   
566     99       U.S.          1         US      0  Cross-industry   
269     99       U.S.          1         US      0  Cross-industry   

             i_group  own_code occ_code  \
835   cross-industry      1235  43-1000   
836   cross-industry      1235  43-1010   
793   cross-industry      1235

##### Observations

There seems to be two types of duplicates here. Some appear to be the same jobs with the same reported metrics, again with differences in the `o_group` column.

However, others appear to be from completely different industries, with different metrics. Therefore, sometiems, different field simply have the same number of total workers.

There are `130` rows, which is quite a few for a manual inspection.

(We also note that the `130` rows is different than the `68` duplicates reported previously. This may mean that some of these duplicates in `tot_emp` are happening at intervals higher than `2`.)

We know from the SOC User Manual that the `occ_code` column gives us information about job categorization. 

One way to go forward could be to group duplicates by those that have similar starting digits in `occ_code`, and then those that do not can be ignored.

#### Add To Do Item

- Isolate duplicates in `tot_emp` by `occ_code`
- Remove duplicates

#### Isolating by `occ_code`

Our first task is to create a filtering dataframe to capture the rows that will be removed from the main dataframe in the end

In [141]:
nat_data_filtered = nat_data[nat_data.groupby('tot_emp')['tot_emp'].transform('count') > threshold].sort_values(by='tot_emp', ascending=False)

In [142]:
nat_data_filtered

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
835,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,43-1000,Supervisors of Office and Administrative Suppo...,...,31.8,39.59,49.51,43920,53190,66140,82340,102980,NaN,NaN
836,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,43-1010,First-Line Supervisors of Office and Administr...,...,31.8,39.59,49.51,43920,53190,66140,82340,102980,NaN,NaN
793,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,41-1010,First-Line Supervisors of Sales Workers,...,23.81,32.19,45.76,32630,38710,49510,66960,95170,NaN,NaN
792,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,41-1000,Supervisors of Sales Workers,...,23.81,32.19,45.76,32630,38710,49510,66960,95170,NaN,NaN
815,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,41-3091,"Sales Representatives of Services, Except Adve...",...,31.86,47.49,68.29,36930,47120,66260,98780,142040,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1005,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,47-2142,Paperhangers,...,23.2,28.11,33.4,35020,40930,48260,58470,69470,NaN,NaN
844,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,43-2099,"Communications Equipment Operators, All Other",...,24,30.92,42.84,35980,44620,49910,64310,89110,NaN,NaN
843,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,43-2090,Miscellaneous Communications Equipment Operators,...,24,30.92,42.84,35980,44620,49910,64310,89110,NaN,NaN
566,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-1243,Pediatric Surgeons,...,#,#,#,189720,#,#,#,#,NaN,NaN


Let's remove columns that we don't need for this portion of the wrangling process.

In [143]:
col_lim_tot_emp = [
    'area',
    'area_title', 
    'area_type',
    'prim_state',
    'naics',
    'naics_title',
    'i_group',
    'own_code',
    'mean_prse',
    'h_pct10',
    'h_pct25',
    'h_median',
    'h_pct75', 
    'h_pct90',
    'a_pct10',
    'a_pct25',
    'a_median',
    'a_pct75', 
    'a_pct90',
    'annual',
    'hourly'
]

In [144]:
nat_data_filtered = nat_data_filtered.drop(columns=col_lim_tot_emp)

In [145]:
nat_data_filtered

,occ_code,occ_title,o_group,tot_emp,emp_prse,h_mean,a_mean
835,43-1000,Supervisors of Office and Administrative Suppo...,minor,1495580,0.4,34.4,71560
836,43-1010,First-Line Supervisors of Office and Administr...,broad,1495580,0.4,34.4,71560
793,41-1010,First-Line Supervisors of Sales Workers,broad,1332180,0.4,28.77,59830
792,41-1000,Supervisors of Sales Workers,minor,1332180,0.4,28.77,59830
815,41-3091,"Sales Representatives of Services, Except Adve...",detailed,1189330,0.7,39.07,81260
...,...,...,...,...,...,...,...
1005,47-2142,Paperhangers,detailed,1520,13.7,24.95,51900
844,43-2099,"Communications Equipment Operators, All Other",detailed,1390,9.9,26.87,55890
843,43-2090,Miscellaneous Communications Equipment Operators,broad,1390,9.9,26.87,55890
566,29-1243,Pediatric Surgeons,detailed,1050,23.4,216.74,450810


#### Compare by `occ_code` Starting Digits

Looking at the SOC User Guide we can see details about the `occ_code` string meanings.

> Occupational Coding
>
> Each item in the hierarchy is designated by a six-digit code. The hyphen between the second and third digit is used only for presentation clarity. The first two digits of the SOC code represent the major group; the third digit represents the minor group; the fourth and fifth digits represent the broad occupation; and the detailed occupation is represented by the sixth digit. Major group codes end with 0000 (e.g., 33-0000, Protective Service Occupations), minor groups end with 000 (e.g., 33-2000, Fire Fighting Workers), and broad occupations end with 0 (e.g., 33-2020, Fire Inspectors). All residuals ("Other," "Miscellaneous," or "All Other"), whether at the detailed or broad occupation or minor group level, contain a 9 at the level of the residual. Detailed residual occupations end in 9 (e.g., 33-9199, Protective Service Workers, All Other); broad occupations which are minor group residuals end in 90 (e.g., 33-9190, Miscellaneous Protective Service Workers); and minor groups which are major group residuals end in 9000 (e.g., 33-9000, Other Protective Service Workers):
> 33-0000  Protective Service Occupations
> > 33-9000  Other Protective Service Workers
> > > 33-9190  Miscellaneous Protective Service Workers
> > > > 33-9199  Protective Service Workers, All Other

By setting the string depth to `5`, we can narrow down our results to only those which have the same number in the first digit of the `broad` category.

In [146]:
str_depth = 5

In [147]:
# From the filtered dataset of duplicates, capture the count of the broad-level categories
# Return the series as a dataframe
first_str = pd.DataFrame(nat_data_filtered['occ_code'].str[:str_depth].value_counts())

In [148]:
with pd.option_context('display.max_rows', None):
    print(first_str)

          count
occ_code       
27-30         4
51-40         4
49-90         4
17-21         4
19-20         3
27-20         3
19-10         3
47-50         3
25-30         3
43-91         3
27-40         3
51-70         3
15-12         3
19-40         3
11-91         3
19-30         3
29-12         3
37-10         3
45-10         2
43-30         2
29-10         2
43-10         2
53-71         2
53-30         2
51-91         2
21-20         2
53-60         2
27-10         2
15-20         2
53-40         2
17-30         2
43-20         2
41-10         2
47-10         2
39-90         2
53-10         2
13-20         2
51-10         2
25-90         2
49-10         2
35-90         2
51-41         2
13-11         2
41-30         2
43-41         2
51-80         1
33-90         1
39-40         1
47-21         1
51-20         1
51-30         1
47-20         1
25-20         1
11-30         1
17-20         1
35-20         1
53-50         1
51-90         1
29-90         1
21-10         1
17-10   

#### Observe `tot_emp` Duplicates with Only One `occ_code` Instance Each 

Those that have a count of `1` are unlikely to be duplicates. Let's review those first. 

In [149]:
# From the list of rows of the broad-level occ_code's
# capture those that only appear once
# Return the index of the resulting series as a dataframe 
first_str_lim = pd.DataFrame(first_str[first_str['count'] == threshold].index)

In [150]:
first_str_lim.sort_values(by='occ_code', ascending=False)

,occ_code
11,53-50
12,51-90
0,51-80
5,51-30
4,51-20
3,47-21
6,47-20
16,41-90
2,39-40
10,35-20


In [151]:
# Create a dataframe to hold a list of the appropriate length broad_level rows
broad_level_list = pd.DataFrame(nat_data_filtered['occ_code'].str[:str_depth])

In [152]:
# Re-filter nat_data_filtered to capture only those rows where the broad-level code appears once 
broad_level_list_one = nat_data_filtered[broad_level_list['occ_code'].isin(first_str_lim['occ_code'])].sort_values(by='occ_code')

In [153]:
broad_level_list_one

,occ_code,occ_title,o_group,tot_emp,emp_prse,h_mean,a_mean
21,11-3013,Facilities Managers,detailed,141090,0.7,55.06,114520
47,11-9039,"Education Administrators, All Other",detailed,53330,1.9,47.82,99460
173,17-1020,"Surveyors, Cartographers, and Photogrammetrists",broad,65870,2.4,37.46,77920
183,17-2040,Chemical Engineers,broad,20330,3.7,61.75,128430
312,21-1013,Marriage and Family Therapists,detailed,65870,4.0,34.96,72720
406,25-2023,"Career/Technical Education Teachers, Middle Sc...",detailed,14200,3.6,*,68690
609,29-9099,Healthcare Practitioners and Technical Workers...,detailed,36970,1.9,35.19,73200
672,33-9031,Gambling Surveillance Officers and Gambling In...,detailed,10000,2.1,22.46,46710
692,35-2019,"Cooks, All Other",detailed,23590,7.4,18.27,38000
760,39-4012,Crematory Operators,detailed,2950,5.7,21.53,44790


In [154]:
broad_level_list_one.shape

(18, 7)

##### Analysis 

The above results show that we have isolated the rows that have:
- duplicated `tot_emp` values
- only one broad-level `occ_code` instance

#### Update To Do Items

- ~~Isolate duplicates in `tot_emp` by `occ_code`~~
- Analyze duplicates
- Remove duplicates

#### Compare Subgroups to Each Other

Let's check and see which rows in the overall `130` `nat_data_filtered` dataset have `tot_emp` values that match the above set.

In [155]:
# Look in the first and second filtered df's to see where the 'tot_emp' values match
nat_data_filtered[nat_data_filtered['tot_emp'].isin(broad_level_list_one['tot_emp'])].sort_values(by='tot_emp', ascending=False)

,occ_code,occ_title,o_group,tot_emp,emp_prse,h_mean,a_mean
1170,51-3022,"Meat, Poultry, and Fish Cutters and Trimmers",detailed,141090,1.2,18.58,38640
21,11-3013,Facilities Managers,detailed,141090,0.7,55.06,114520
1352,53-5000,Water Transportation Workers,minor,77710,2.4,38.54,80150
1206,51-4190,Miscellaneous Metal Workers and Plastic Workers,broad,77710,1.8,22.84,47500
312,21-1013,Marriage and Family Therapists,detailed,65870,4.0,34.96,72720
173,17-1020,"Surveyors, Cartographers, and Photogrammetrists",broad,65870,2.4,37.46,77920
822,41-9011,Demonstrators and Product Promoters,detailed,64770,5.4,21.03,43730
145,15-1243,Database Architects,detailed,64770,2.7,68.57,142620
503,27-4030,"Television, Video, and Film Camera Operators a...",broad,53330,3.8,39.35,81850
47,11-9039,"Education Administrators, All Other",detailed,53330,1.9,47.82,99460


In [156]:
nat_data_filtered[nat_data_filtered['tot_emp'].isin(broad_level_list_one['tot_emp'])].shape

(35, 7)

Of the `35` rows, most appear to not be duplicates. 

However, a visual scan shows some issues. Note that `Miscellaneous Rail Transportation Workers` and `Rail Transportation Workers, All Other` are duplicates, as are religious workers.

#### Compare Against `emp_prse` Instead

We see when we look at these we see that, in both instances, the associated `emp_prse` are all equal. (This is also true for other columns, such as `h_mean` and `a_mean`.)

We can use this to further limit our duplicates at this point:
- within a grouping by `tot_emp` duplicates, if the values in `emp_prse` do not also match, they're not duplicates.

The filtered dataset is supposed to be a list of duplicates, therefore these non-duplicate rows should be dropped at this stage.

#### Update To Do Items

- ~~Isolate duplicates in `tot_emp` by `occ_code`~~
- Isolate duplicates in `tot_emp` by `emp_prse`
- Analyze duplicates
- Remove duplicates

#### Create a Custom Function

In [157]:
# Create temporary drop and count markers for the function
count_marker = 'ct'
drop_marker = 'drop'

In [158]:
# Add the temporary columns to the filtered dataset

nat_data_filtered[count_marker] = 0
nat_data_filtered[drop_marker] = False

In [159]:
# Checks a df first by initial grouping, and then subgrouping to discern duplicates
# Updates provided df in place
# Requires initial dataframe have columns for marking counting within subgroups and a drop marker
def check_unique_within_subgroup(
    df,
    fir_grp,
    sec_grp,
    threshold,
    count_marker,
    drop_marker
):    
    # Iterate through the first group and separate by fir_grp
    for grp_num, group_df in df.groupby(fir_grp):
        
        # Debug:
        # print(f"Checking group: {grp_num}")

        # Iterate through sec_grp, count values, and set count of each to count marker column
        group_df[count_marker] = group_df.groupby(sec_grp)[sec_grp].transform('count')
        
        # In the drop_marker col, indicate whether this passes the threshold
        group_df[drop_marker] = group_df[count_marker] == threshold
        # print(group_df)

        # For loop to iterate through each value in the subgroup
        # and, in the parent df, set the associated count_marker and drop_marker
        for grp_idx, grp_row in group_df.iterrows():

            # Debug:
            # print(f"\ngrp_idx: {grp_idx},\ncount_marker: {grp_row[count_marker]},\ndrop_marker: {grp_row[drop_marker]}")
            
            # Debug:
            # print(f"\ndf tot_emp: {df.loc[grp_idx, fir_grp]}\ndf curr count: {df.loc[grp_idx, count_marker]}")

            # Set the count_marker in the parent df
            df.loc[grp_idx, count_marker] = grp_row[count_marker]

            # Set the drop_marker in the parent df
            df.loc[grp_idx, drop_marker] = grp_row[drop_marker]

#### Call the Custom Function

In [160]:
# Call the custom function on nat_data_filtered
check_unique_within_subgroup(nat_data_filtered, 'tot_emp', 'emp_prse', threshold, count_marker, drop_marker) 

In [161]:
# Check result
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 32):
    print(nat_data_filtered[['occ_title', 'emp_prse', 'tot_emp', count_marker, drop_marker, 'a_mean']])

                            occ_title  emp_prse  tot_emp  ct   drop  a_mean
835   Supervisors of Office and Ad...       0.4  1495580   2  False   71560
836   First-Line Supervisors of Of...       0.4  1495580   2  False   71560
793   First-Line Supervisors of Sa...       0.4  1332180   2  False   59830
792      Supervisors of Sales Workers       0.4  1332180   2  False   59830
815   Sales Representatives of Ser...       0.7  1189330   2  False   81260
814   Miscellaneous Sales Represen...       0.7  1189330   2  False   81260
106   Miscellaneous Business Opera...       0.6  1128200   2  False   92380
107   Business Operations Speciali...       0.6  1128200   2  False   92380
968   Supervisors of Construction ...       0.6   806080   2  False   84500
969   First-Line Supervisors of Co...       0.6   806080   2  False   84500
1147  First-Line Supervisors of Pr...       0.3   685140   2  False   74540
1146  Supervisors of Production Wo...       0.3   685140   2  False   74540
73          

#### Verification

A visual inspection shows that the logic appears to be functioning properly.

Duplicates are found for each row that has:
- The same `tot_emp` value
- The same `emp_prse` value

We can see that the `a_mean` values also match.

We are prepared to remove from the filtering dataset all values that have a count of `1`, because they are not duplicates.

#### Prepare for Drop

Check that all values that return `True` in the `drop_marker` column are not duplicates.

We want to remove these from the filtered dataset, so that they will stay when we later remove the filtered dataset from the main df.

We want to check that when we return `drop_marker == True` values, either:
- The `tot_emp` value prints only once
- Or, if the `tot_emp` is a duplicate, the associated `emp_prse` values do not match

In [162]:
with pd.option_context('display.max_rows',None):
    print(nat_data_filtered[nat_data_filtered[drop_marker] == True].sort_values(by='tot_emp',ascending=False))

     occ_code                                          occ_title   o_group  \
1188  51-4040                                         Machinists     broad   
423   25-3040                                             Tutors     broad   
716   37-1011  First-Line Supervisors of Housekeeping and Jan...  detailed   
1170  51-3022       Meat, Poultry, and Fish Cutters and Trimmers  detailed   
21    11-3013                                Facilities Managers  detailed   
491   27-3090      Miscellaneous Media and Communication Workers     broad   
1241  51-7010                 Cabinetmakers and Bench Carpenters     broad   
1126  49-9060       Precision Instrument and Equipment Repairers     broad   
1352  53-5000                       Water Transportation Workers     minor   
1206  51-4190    Miscellaneous Metal Workers and Plastic Workers     broad   
312   21-1013                     Marriage and Family Therapists  detailed   
173   17-1020    Surveyors, Cartographers, and Photogrammetrists

In [163]:
nat_data_filtered[nat_data_filtered[drop_marker] == True].shape

(62, 9)

In [164]:
nat_data_filtered[nat_data_filtered[drop_marker] == True]['tot_emp'].value_counts()

tot_emp
14340     2
174660    2
2950      2
3430      2
5720      2
5730      2
5900      2
8440      2
9060      2
9680      2
10000     2
12170     2
12500     2
13810     2
14200     2
1050      2
14370     2
53330     2
141090    2
79540     2
77710     2
65870     2
18970     2
64770     2
41890     2
36970     2
28550     2
23590     2
20330     2
6590      1
89580     1
1520      1
298790    1
Name: count, dtype: int64

Note that there are `4` values that are not have matching pairs in this subset. (We could guess that they are part of a set of `3` or another odd number, where the other rows are considered duplicated.)

Therefore, while there are `62` rows that we want to keep in our final `nat_data` dataset, only `58` of the values from this current subset will be considered duplicated. We should expect to see `29` as the returned value from `...duplicated().sum()` at the final stage. 

#### Remove Non-Duplicates From Filtered Dataframe

In [165]:
nat_data_filtered = nat_data_filtered[nat_data_filtered[drop_marker] == False]

In [166]:
nat_data_filtered

,occ_code,occ_title,o_group,tot_emp,emp_prse,h_mean,a_mean,ct,drop
835,43-1000,Supervisors of Office and Administrative Suppo...,minor,1495580,0.4,34.4,71560,2,False
836,43-1010,First-Line Supervisors of Office and Administr...,broad,1495580,0.4,34.4,71560,2,False
793,41-1010,First-Line Supervisors of Sales Workers,broad,1332180,0.4,28.77,59830,2,False
792,41-1000,Supervisors of Sales Workers,minor,1332180,0.4,28.77,59830,2,False
815,41-3091,"Sales Representatives of Services, Except Adve...",detailed,1189330,0.7,39.07,81260,2,False
...,...,...,...,...,...,...,...,...,...
167,15-2099,"Mathematical Science Occupations, All Other",detailed,4660,6.5,40.72,84700,2,False
1350,53-4090,Miscellaneous Rail Transportation Workers,broad,1520,10.3,25.72,53500,2,False
1351,53-4099,"Rail Transportation Workers, All Other",detailed,1520,10.3,25.72,53500,2,False
844,43-2099,"Communications Equipment Operators, All Other",detailed,1390,9.9,26.87,55890,2,False


#### Remove Rows in Filtered Dataframe from Main National Dataframe 

In [167]:
nat_data_pre_drop = nat_data.drop(nat_data_filtered.index)

In [168]:
nat_data_pre_drop.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
5,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1020,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN


In [169]:
nat_data_pre_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1070 entries, 0 to 1399
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1070 non-null   int64  
 1   area_title   1070 non-null   object 
 2   area_type    1070 non-null   int64  
 3   prim_state   1070 non-null   object 
 4   naics        1070 non-null   int64  
 5   naics_title  1070 non-null   object 
 6   i_group      1070 non-null   object 
 7   own_code     1070 non-null   int64  
 8   occ_code     1070 non-null   object 
 9   occ_title    1070 non-null   object 
 10  o_group      1070 non-null   object 
 11  tot_emp      1070 non-null   int64  
 12  emp_prse     1070 non-null   float64
 13  h_mean       1070 non-null   object 
 14  a_mean       1070 non-null   object 
 15  mean_prse    1070 non-null   float64
 16  h_pct10      1070 non-null   object 
 17  h_pct25      1070 non-null   object 
 18  h_median     1070 non-null   object 
 19  h_pct75    

In [170]:
nat_data_pre_drop.shape

(1070, 28)

Previously, we had `1138` rows. We saw that `nat_data_filtered` resulted in `68` rows for deletion. Now, we have `1070` rows.

In [171]:
1138 - 68

1070

In [172]:
nat_data_pre_drop['tot_emp'].duplicated().sum()

np.int64(29)

There is our expected value of `29`.

In [173]:
nat_data_pre_drop[nat_data_pre_drop['tot_emp'].duplicated() == True]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
312,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,21-1013,Marriage and Family Therapists,...,30.66,40.87,53.66,42610,48600,63780,85020,111610,NaN,NaN
406,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,25-2023,"Career/Technical Education Teachers, Middle Sc...",...,*,*,*,47090,55920,63620,78270,98430,True,NaN
468,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,27-2021,Athletes and Sports Competitors,...,*,*,*,24960,36750,62360,130770,#,True,NaN
472,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,27-2031,Dancers,...,23.97,37.43,53.66,*,*,*,*,*,NaN,True
503,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,27-4030,"Television, Video, and Film Camera Operators a...",...,33.93,49.19,64.83,37600,49420,70570,102310,134840,NaN,NaN
550,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-1211,Anesthesiologists,...,#,#,#,124450,186680,#,#,#,NaN,NaN
566,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-1243,Pediatric Surgeons,...,#,#,#,189720,#,#,#,#,NaN,NaN
569,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-1291,Acupuncturists,...,37.57,51.5,76.22,41840,54090,78140,107120,158540,NaN,NaN
609,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-9099,Healthcare Practitioners and Technical Workers...,...,30.78,43.75,61.22,37220,45250,64030,91000,127340,NaN,NaN
672,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,33-9031,Gambling Surveillance Officers and Gambling In...,...,21.11,24.33,29.98,34020,37410,43900,50610,62360,NaN,NaN


We see that our data is prepared to be shifted to the main df.

In [174]:
nat_data = nat_data_pre_drop

#### Remove To Do Items

- ~~Isolate duplicates in `tot_emp` by `occ_code`~~
- ~~Isolate duplicates in `tot_emp` by `emp_prse`~~
- ~~Analyze duplicates~~
- ~~Remove duplicates~~

## `emp_prse` Wrangle

The `emp_prse` column is numerical.

We won't be able to tell much about duplicates, but we can simply check over the df.

#### Update To Do Items

- Analyze and inspect `emp_prse`

In [175]:
list_unique = nat_data['emp_prse'].unique()

In [176]:
temp_df = pd.DataFrame()
temp_df['emp_prse_count'] = pd.DataFrame(list_unique)

In [177]:
temp_df.sort_index(ascending=False)

,emp_prse_count
129,19.0
128,22.4
127,12.5
126,38.4
125,13.0
...,...
4,0.4
3,1.2
2,0.3
1,0.2


In [178]:
temp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 1 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   emp_prse_count  130 non-null    float64
dtypes: float64(1)
memory usage: 1.1 KB


In [179]:
with pd.option_context('max_colwidth', None):
    print(nat_desc.loc[nat_desc['Field'] == 'emp_prse', 'Field Description'])

12    Percent relative standard error (PRSE) for the employment estimate. PRSE is a measure of sampling error, expressed as a percentage of the corresponding estimate. Sampling error occurs when values for a population are estimated from a sample survey of the population, rather than calculated from data for all members of the population. Estimates with lower PRSEs are typically more precise in the presence of sampling error.
Name: Field Description, dtype: object


##### Observations on `emp_prse`

This seems straight forward and needs no further intervention.

#### Update To Do Items

- ~~Analyze and inspect `emp_prse`~~

## `mean_prse` Wrangle

We're going to go out of order and do `mean_prse` next, because the annual and hourly-related columns can all be grouped together for simultaneous analysis.

The analysis here is similar to `emp_prse`.

#### Update To Do Items

- Analyze and inspect `mean_prse`

In [180]:
list_unique = nat_data['emp_prse'].unique()

In [181]:
temp_df = pd.DataFrame()
temp_df['mean_prse_count'] = pd.DataFrame(list_unique)

In [182]:
temp_df.sort_index(ascending=False)

,mean_prse_count
129,19.0
128,22.4
127,12.5
126,38.4
125,13.0
...,...
4,0.4
3,1.2
2,0.3
1,0.2


In [183]:
temp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 1 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   mean_prse_count  130 non-null    float64
dtypes: float64(1)
memory usage: 1.1 KB


In [184]:
with pd.option_context('max_colwidth', None):
    print(nat_desc.loc[nat_desc['Field'] == 'mean_prse', 'Field Description'])

19    Percent relative standard error (PRSE) for the mean wage estimate. PRSE is a measure of sampling error, expressed as a percentage of the corresponding estimate. Sampling error occurs when values for a population are estimated from a sample survey of the population, rather than calculated from data for all members of the population. Estimates with lower PRSEs are typically more precise in the presence of sampling error.
Name: Field Description, dtype: object


In [185]:
temp_df.sort_index(ascending=False)

,mean_prse_count
129,19.0
128,22.4
127,12.5
126,38.4
125,13.0
...,...
4,0.4
3,1.2
2,0.3
1,0.2


The `mean_prse` column is straightforward as well. There are no `0` or `Null` values.

#### Update To Do Items

- ~~Analyze and inspect `mean_prse`~~

## Hourly and Annual Data Analysis

#### Initial Analysis and Grouping of Columns

We will analyze the hourly and annual-related columns as one group, first.

In [186]:
col_hr_an = [
    'h_mean',
    'a_mean',
    'mean_prse',
    'h_pct10',
    'h_pct25',
    'h_median',
    'h_pct75',
    'h_pct90',
    'a_pct10',
    'a_pct25',
    'a_median',
    'a_pct75',
    'a_pct90',
    'annual',
    'hourly'
]

In [187]:
nat_data[col_hr_an].info()

<class 'pandas.core.frame.DataFrame'>
Index: 1070 entries, 0 to 1399
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   h_mean     1070 non-null   object 
 1   a_mean     1070 non-null   object 
 2   mean_prse  1070 non-null   float64
 3   h_pct10    1070 non-null   object 
 4   h_pct25    1070 non-null   object 
 5   h_median   1070 non-null   object 
 6   h_pct75    1070 non-null   object 
 7   h_pct90    1070 non-null   object 
 8   a_pct10    1070 non-null   object 
 9   a_pct25    1070 non-null   object 
 10  a_median   1070 non-null   object 
 11  a_pct75    1070 non-null   object 
 12  a_pct90    1070 non-null   object 
 13  annual     78 non-null     object 
 14  hourly     7 non-null      object 
dtypes: float64(1), object(14)
memory usage: 133.8+ KB


In [188]:
nat_data[col_hr_an].head(10)

,h_mean,a_mean,mean_prse,h_pct10,h_pct25,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,32.66,67920,0.1,14.42,17.66,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,68.15,141760,0.2,27.41,38.42,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,67.24,139860,0.4,22.84,33.08,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,126.41,262930,0.9,35.44,60.61,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
5,64,133120,0.4,22.8,32.29,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
7,*,67390,1.6,*,*,*,*,*,20380,29120,44810,80350,137820,True,NaN
9,78.72,163740,0.3,35.25,48.43,69.48,98.52,#,73320,100740,144530,204920,#,NaN,NaN
10,71.76,149270,2.3,30.29,41.34,61.04,85.85,#,63000,85990,126960,178570,#,NaN,NaN
12,79.35,165050,0.3,35.13,48.65,71.57,99.24,#,73080,101200,148870,206410,#,NaN,NaN
13,82.46,171520,0.6,39.38,53.47,77.42,101.48,#,81900,111210,161030,211080,#,NaN,NaN


In [189]:
nat_data['h_pct90'][0]

60.44

In [190]:
print(type(nat_data['h_pct90'][0]))

<class 'float'>


In [191]:
nat_data['h_pct90'][1]

'#'

In [192]:
print(type(nat_data['h_pct90'][1]))

<class 'str'>


##### Commentary

We see that these columns have mixed types. For each of the columns in each dataset, we need to resolve this issue.

The majority of the `annual` and `hourly` columns are `Null`. We will need to inspect and decide what this data means.

Let's focus on the `h_pct90` column, picked at random.

The first task is to find out how many types there are.

The second task is to find out how many unique types of strings there are.

#### Update To Do Items

- Analyze types and particulars of strings
- Replace or drop rows across all columns as necessary
- Analyze and decide on course of action for `hourly` and `annual` columns

In [193]:
hr_an_desc = pd.DataFrame(nat_desc.loc[nat_desc['Field'].isin(col_hr_an),['Field', 'Field Description']])

In [194]:
with pd.option_context('max_colwidth', None):
    print(hr_an_desc)

        Field  \
17     h_mean   
18     a_mean   
19  mean_prse   
20    h_pct10   
21    h_pct25   
22   h_median   
23    h_pct75   
24    h_pct90   
25    a_pct10   
26    a_pct25   
27   a_median   
28    a_pct75   
29    a_pct90   

                                                                                                                                                                                                                                                                                                                                                                                                                         Field Description  
17                                                                                                                                                                                                                                                                                                                                          

Everything here is straightforward.

#### Reviewing the Notes Dataframe

Now, we need to also look at the `notes_df` from earlier.

In [195]:
with pd.option_context('max_colwidth', None):
    print(notes_df)

                                                                                         Notes
34                                        *  = indicates that a wage estimate is not available
35                                **  = indicates that an employment estimate is not available
36       #  = indicates a wage equal to or greater than $115.00 per hour or $239,200 per year 
37  ~ =indicates that the percent of establishments reporting the occupation is less than 0.5%


#### Observing the Various Types Within One Row

Let's look at the various types that are found within one column, as a representation of the rest.

In [196]:
nat_data['h_pct90'].apply(type).unique()

array([<class 'float'>, <class 'str'>, <class 'int'>], dtype=object)

We see `float`, `str`, `int`. 

The `float` values are probably desirable data points and don't need to be changed.

The `str` values will be those values found in the notes.

The `int` values need further inspection. Likely, they are `float`'s that were cast into `int`'s during the import stage, or something similar.

#### Value Counting

In [197]:
nat_data['h_pct90'][nat_data['h_pct90'].apply(type) == str].unique()

array(['#', '*'], dtype=object)

In [198]:
nat_data['h_pct90'][nat_data['h_pct90'].apply(type) == int].unique()

array([110, 86, 61, 40, 60, 50, 20, 29, 54, 23, 35, 33], dtype=object)

In [199]:
nat_data['h_pct90'][nat_data['h_pct90'].apply(type) == int].value_counts()

h_pct90
40     4
110    1
86     1
61     1
60     1
50     1
20     1
29     1
54     1
23     1
35     1
33     1
Name: count, dtype: int64

#### Analysis for `int` Values

The `int` values all seem straight forward. We will simply cast them back into `float` values.


##### Looking at the Highest Values

In [200]:
nat_num_only = pd.DataFrame(nat_data['h_pct90'][(nat_data['h_pct90'].apply(type) != str)])

In [201]:
nat_num_only['h_pct90'][nat_num_only['h_pct90'] > 99]

17      104.16
28      105.35
36      105.76
46      102.12
57      105.33
117     102.79
137      111.6
145     100.96
147      99.92
149     101.66
156      99.24
187     107.61
207        110
232     113.34
264     102.27
269     107.97
339     101.39
342     104.11
449     101.64
474     100.11
476     105.44
509     102.34
537     102.35
810     103.47
1322    101.16
Name: h_pct90, dtype: object

#### Analysis for `str` Values

We have a conundrum.

Regarding the `#` value, this `str` is prevalant in many of the columns.

The representation is important and cannot be dropped. At the same time, it is non-specific.

What I will do for now is simply insert the represented lowest value of $`115`/hr.

The same will go for the annual values, with the value being $`239,200`/year.

We will need to adjust the method of analysis whenever these values arise. In many cases, these values may simply have to be dropped from consideration.

Since there are a significant number of these values, the rows may not be able to be dropped altogether.

We may need to create two groups: a normal group and a high-income group.

#### Analyzing * Symbols

The next question: what to do about the `*` symbols? 

In [202]:
temp_df = nat_data[nat_data['h_pct90'] == '*']

In [203]:
temp_df[col_hr_an].head()

,h_mean,a_mean,mean_prse,h_pct10,h_pct25,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
7,*,67390,1.6,*,*,*,*,*,20380,29120,44810,80350,137820,True,NaN
45,*,113360,0.4,*,*,*,*,*,72400,83840,104070,132550,165820,True,NaN
350,*,102410,0.5,*,*,*,*,*,47010,61300,81600,125900,180630,True,NaN
351,*,113840,0.6,*,*,*,*,*,46460,63040,97270,140360,210530,True,NaN
353,*,98400,0.6,*,*,*,*,*,48470,62160,82390,124840,168080,True,NaN


In [204]:
type(nat_data.loc[nat_data['h_pct90'] == '*','a_pct10'].values[0])

int

##### Commentary

An initial inspection shows that the hourly columns are at least someimtes blank when the annual columns have values. 

That raises the question, how many instances exist where hourly is `*` and annual is something other than an int?

In [205]:
# Lambda logic to obtain a_pct90 instances of non-int values 
nat_data[nat_data['a_pct90'].apply(lambda x: isinstance(x, int)) == False].head(2)

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN


In [206]:
nat_data[nat_data['a_pct90'].apply(lambda x: isinstance(x, int)) == False].shape

(67, 28)

In [207]:
# Logic to obtain h_pct90 is * and a_pct is not #
(nat_data['h_pct90'] == '*') & (nat_data['a_pct90'] != '#')

0       False
1       False
2       False
3       False
5       False
        ...  
1394    False
1395    False
1396    False
1397    False
1399    False
Length: 1070, dtype: bool

In [208]:
# Test that instances are found
((nat_data['h_pct90'] == '*') & (nat_data['a_pct90'] != '#')).sum()

np.int64(71)

In [209]:
# Combine the two together
nat_data[(nat_data['h_pct90'] == '*') & (nat_data['a_pct90'] != '#') & (nat_data['a_pct90'].apply(lambda x: isinstance(x, int)) == False)]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly


##### Commentary

A preliminary, quick study shows that if the value in an hourly column is `*`, it is possible that no similar value matches in the associated annual column.

This search is needs to be expanded across all hourly and annual columns.

The `h_mean` and `a_mean` columns are likely composites of the two different types of numbers.

Logic stands to reason that we can simplify our search to see wehther these are ever both equal to `*` at the same time. If not, then we can assume, for now, that the two types of columns do not conflict.

#### Comparing `h_mean` and `a_mean` 

In [210]:
nat_data[(nat_data['h_mean'] == '*') & (nat_data['a_mean'] == '*')]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly


##### Commentary

There appears to be no overlap between the two columns.

#### Analyzing the `hourly` and `annual` Columns

The next question is whether there are rows where **both** `hourly` and `annual` values are provided.

In [211]:
nat_data[(nat_data['h_mean'].apply(lambda x: isinstance(x, int)) == True) & (nat_data['a_mean'].apply(lambda x: isinstance(x, int)) == True)].shape

(10, 28)

In [212]:
temp_df = nat_data[(nat_data['h_mean'].apply(lambda x: isinstance(x, int)) == True) & (nat_data['a_mean'].apply(lambda x: isinstance(x, int)) == True)]

In [213]:
temp_col = ['occ_title'] + col_hr_an
temp_df[temp_col]

,occ_title,h_mean,a_mean,mean_prse,h_pct10,h_pct25,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
5,General and Operations Managers,64,133120,0.4,22.8,32.29,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
189,Electrical and Electronics Engineers,60,124810,0.6,36.63,44.42,57.11,72.37,87.21,76190,92390,118780,150530,181390,NaN,NaN
346,Miscellaneous Legal Support Workers,35,72800,1.0,18.35,22.57,29.11,39.13,62.75,38170,46950,60550,81390,130510,NaN,NaN
791,Sales and Related Occupations,26,54070,0.3,13.01,14.91,18.01,28.79,47.53,27060,31010,37460,59880,98860,NaN,NaN
920,Legal Secretaries and Administrative Assistants,29,60320,2.6,17.08,20.54,26.03,34.66,42.15,35530,42720,54140,72090,87660,NaN,NaN
924,Data Entry and Information Processing Workers,21,43680,0.5,15,17.19,19.94,23.7,28.47,31200,35760,41480,49300,59220,NaN,NaN
950,Animal Breeders,27,56150,5.4,17.85,20.82,25,28.67,43.26,37130,43310,52000,59630,89970,NaN,NaN
1121,Millwrights,33,68640,1.0,21.68,26.65,31.33,38.74,45.07,45100,55420,65170,80580,93740,NaN,NaN
1266,"Plant and System Operators, All Other",31,64490,1.6,19.35,23.01,29.67,36.98,44.59,40250,47860,61710,76910,92750,NaN,NaN
1385,Industrial Truck and Tractor Operators,23,47830,0.2,17.55,19.13,22.3,25.81,29.59,36500,39780,46390,53680,61540,NaN,NaN


The answer appears to be, yes. There are `10` rows where both annual and hourly data is provided.

More information will be needed to understand why these industries' parameters allow for both types of pay.

#### Find Instances of ~ and **

In [214]:
nat_data_mask = nat_data[nat_data.isin(['~']).any(axis=1)]

In [215]:
nat_data_mask.shape

(0, 28)

In [216]:
nat_data_max = nat_data[nat_data.isin(['**']).any(axis=1)]

In [217]:
nat_data_mask.shape

(0, 28)

There appears to be no instances of these two symbols in the National Dataset

#### Update To Do Items

- ~~Analyze types and particulars of strings~~
- Replace or drop rows across all columns as necessary:
    - In all hourly data rows, insert `115` for all `#`
    - In all annual data rows, insert `239,200` for all `#`
    - In all annual and hourly rows, insert `null` for all `*`
- Analyze and decide on course of action for `hourly` and `annual` columns
- Research issue where 10 rows have both annual and hourly data

## `h_mean`

The `h_mean` column contains `*` strings that need to be converted to `NaN`, and `int` values that need to be converted to `float`.

### Converting str to NaN

In [218]:
# List of columns to drop from the filtered dataset for simplicity
an_hr_col_drop = [
    'area',
    'area_title',
    'area_type',
    'prim_state',
    'naics',
    'naics_title',
    'i_group',
    'own_code',
    'occ_code',
    'o_group',
    'tot_emp',
    'emp_prse'
]

In [219]:
# Filter the primary dataset
nat_data_filtered = nat_data.drop(columns=an_hr_col_drop, axis=1)

In [220]:
# Capture a map of the types in the h_mean column
h_mean_df = pd.DataFrame(nat_data_filtered['h_mean'].map(type))

In [221]:
# Capture the index of all rows of h_mean that are of type str
h_mean_str_idx = h_mean_df[h_mean_df['h_mean'] == str].index

In [222]:
# Capture all str values in the h_mean col
nat_data_h_mean_str = pd.DataFrame(nat_data_filtered['h_mean'].loc[h_mean_str_idx])

In [223]:
nat_data_h_mean_str['h_mean'].value_counts()

h_mean
*    78
Name: count, dtype: int64

There are `78` instances of `*` to be adjusted. There are no `#` instances.

In [224]:
# Capture the index of all `*` cells
h_mean_star_idx = nat_data_h_mean_str[nat_data_h_mean_str['h_mean'] == '*'].index

In [225]:
print(len(h_mean_star_idx))

78


In [226]:
# Set each of these cells to `NaN`
nat_data_filtered.loc[h_mean_star_idx, 'h_mean'] = np.nan

In [227]:
# Check that no type str values remain in the column
nat_data_filtered[nat_data_filtered['h_mean'].apply(type) == str].shape

(0, 16)

In [228]:
# Check that each of the targeted cells is NaN 
nat_data_filtered['h_mean'].loc[h_mean_star_idx]

7       NaN
45      NaN
350     NaN
351     NaN
353     NaN
       ... 
1317    NaN
1318    NaN
1319    NaN
1320    NaN
1324    NaN
Name: h_mean, Length: 78, dtype: object

### Set all int to float

Now that all of the `str` values are set to `NaN`, all `int` values can be easily converted to `float`.

In [229]:
nat_data_filtered['h_mean'] = nat_data_filtered['h_mean'].astype(float)

In [230]:
nat_data_filtered['h_mean'].describe()

count    992.000000
mean      36.408649
std       23.397692
min       14.820000
25%       22.847500
50%       30.115000
75%       41.217500
max      216.740000
Name: h_mean, dtype: float64

In [231]:
nat_data_filtered['h_mean'].isna().sum()

np.int64(78)

### Update `nat_data`

The primary `nat_data` dataset `h_mean` column is prepared to receive the filtered set's wrangled column.

In [232]:
nat_data['h_mean'] = nat_data_filtered['h_mean']

In [233]:
nat_data[nat_data['h_mean'].apply(type) == str].shape

(0, 28)

In [234]:
nat_data[nat_data['h_mean'].apply(type) == int].shape

(0, 28)

In [235]:
nat_data['h_mean'].isna().sum()

np.int64(78)

In [236]:
nat_data['h_mean'].describe()

count    992.000000
mean      36.408649
std       23.397692
min       14.820000
25%       22.847500
50%       30.115000
75%       41.217500
max      216.740000
Name: h_mean, dtype: float64

## a_mean

The `a_mean` column contains `*` strings that need to be converted to `NaN`, and `int` values that need to be converted to `float`.

### Converting `*` to `str`

In [237]:
# Filter the primary dataset
nat_data_filtered = nat_data.drop(columns=an_hr_col_drop, axis=1)

In [238]:
# Capture a map of the types in the a_mean column
a_mean_df = pd.DataFrame(nat_data_filtered['a_mean'].map(type))

In [240]:
# Capture the index of all rows of a_mean that are of type str
a_mean_str_idx = a_mean_df[a_mean_df['a_mean'] == str].index

In [241]:
# Capture all str values in the a_mean col
nat_data_a_mean_str = pd.DataFrame(nat_data_filtered['a_mean'].loc[a_mean_str_idx])

In [242]:
nat_data_a_mean_str['a_mean'].value_counts()

a_mean
*    7
Name: count, dtype: int64

There are `7` instances of `*` and no instances of `#`.

In [243]:
# Capture the index of all `*` cells
a_mean_star_idx = nat_data_a_mean_str[nat_data_a_mean_str['a_mean'] == '*'].index

In [244]:
print(len(a_mean_star_idx))

7


In [245]:
# Set each of these cells to `NaN`
nat_data_filtered.loc[a_mean_star_idx, 'a_mean'] = np.nan

In [246]:
# Check that no type str values remain in the column
nat_data_filtered[nat_data_filtered['a_mean'].apply(type) == str].shape

(0, 16)

In [247]:
# Check that each of the targeted cells is NaN 
nat_data_filtered['a_mean'].loc[a_mean_star_idx]

465    NaN
472    NaN
474    NaN
476    NaN
477    NaN
478    NaN
479    NaN
Name: a_mean, dtype: object

### Set all int to float

Now that all of the `str` values are set to `NaN`, all `int` values can be easily converted to `float`.

In [248]:
nat_data_filtered['a_mean'] = nat_data_filtered['a_mean'].astype(float)

In [249]:
nat_data_filtered['a_mean'].describe()

count      1063.000000
mean      77213.170273
std       48508.978729
min       30830.000000
25%       48560.000000
50%       64220.000000
75%       89740.000000
max      450810.000000
Name: a_mean, dtype: float64

In [250]:
nat_data_filtered['a_mean'].isna().sum()

np.int64(7)

### Update `nat_data`

The primary `nat_data` dataset `h_mean` column is prepared to receive the filtered set's wrangled column.

In [251]:
nat_data['a_mean'] = nat_data_filtered['a_mean']

In [252]:
nat_data[nat_data['a_mean'].apply(type) == str].shape

(0, 28)

In [253]:
nat_data[nat_data['a_mean'].apply(type) == int].shape

(0, 28)

In [254]:
nat_data['a_mean'].isna().sum()

np.int64(7)

In [267]:
nat_data['a_mean'].describe()

count      1063.000000
mean      77213.170273
std       48508.978729
min       30830.000000
25%       48560.000000
50%       64220.000000
75%       89740.000000
max      450810.000000
Name: a_mean, dtype: float64

## mean_prse

### Initial Inspection

Let's first check the description.

In [279]:
with pd.option_context('display.max_colwidth', None):
    print(nat_desc[nat_desc['Field'] == 'mean_prse']['Field Description'])

19    Percent relative standard error (PRSE) for the mean wage estimate. PRSE is a measure of sampling error, expressed as a percentage of the corresponding estimate. Sampling error occurs when values for a population are estimated from a sample survey of the population, rather than calculated from data for all members of the population. Estimates with lower PRSEs are typically more precise in the presence of sampling error.
Name: Field Description, dtype: object


In [258]:
nat_data_filtered['mean_prse'].head()

0    0.1
1    0.2
2    0.4
3    0.9
5    0.4
Name: mean_prse, dtype: float64

The column is of type `float64`, and therefore there are no string values.

In [260]:
nat_data_filtered['mean_prse'].describe()

count    1070.000000
mean        1.224206
std         1.773071
min         0.000000
25%         0.400000
50%         0.700000
75%         1.300000
max        22.100000
Name: mean_prse, dtype: float64

In [261]:
nat_data_filtered['mean_prse'].isna().sum()

np.int64(0)

In [266]:
len(nat_data_filtered[nat_data_filtered['mean_prse'] == 0])

5

In [280]:
nat_data_filtered[nat_data_filtered['mean_prse'] == 0]

,occ_title,h_mean,a_mean,mean_prse,h_pct10,h_pct25,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
61,Postmasters and Mail Superintendents,45.08,93760.0,0.0,39.15,41.9,44.58,47.88,52.47,81430,87150,92730,99590,109140,NaN,NaN
907,Postal Service Workers,28.71,59720.0,0.0,20.38,22.13,27.82,35.6,36.2,42390,46030,57870,74050,75300,NaN,NaN
908,Postal Service Clerks,29.36,61070.0,0.0,20.48,26.64,29.63,35.6,35.6,42600,55410,61630,74050,74050,NaN,NaN
909,Postal Service Mail Carriers,28.79,59880.0,0.0,20.38,22.13,27.64,36.2,36.96,42390,46030,57490,75300,76880,NaN,NaN
910,"Postal Service Mail Sorters, Processors, and P...",28.04,58310.0,0.0,20.48,22.78,27.18,35.08,35.6,42600,47380,56530,72970,74050,NaN,NaN


#### Analysis

Apparently, there is no `mean_prse` sampling error for the Postal service. There may be a legitimate reason.

There is no need to alter this column further.